# Notebook 04: Gemini AI Post Classification

**Detection Vector 4 (Weight: 15% of Manipulation Score)**

This notebook uses Google's Gemini API to perform nuanced classification of Trump's
oil-related posts. Unlike crude keyword matching, Gemini can understand context,
sarcasm, and implicit market signals.

**Why AI classification matters:**
- "Iran's soccer team played great" ≠ "bomb Iran" (keywords fail here)
- Gemini can assess fabrication risk, intent to move markets, and oscillation patterns
- Each post gets a 0-100 manipulation score with reasoning

In [ ]:
import pandas as pd
import json
import time
import os
from google import genai

# Gemini API setup
GEMINI_API_KEY = 'YOUR_GEMINI_KEY_HERE'  # <-- PASTE YOUR KEY
client = genai.Client(api_key=GEMINI_API_KEY)

print('Gemini client initialized.')

In [ ]:
# Load cleaned posts
posts = pd.read_csv('../data/processed/posts_cleaned.csv')
posts['timestamp'] = pd.to_datetime(posts['timestamp'])
posts['date'] = pd.to_datetime(posts['date'])

# Only classify posts from Feb 20, 2026 onward (Iran crisis period)
# The war started Feb 28, so Feb 20 gives us a week of pre-war baseline.
# Earlier posts are used for the master dataset (post counts, volatility baselines)
# but don't need expensive Gemini classification — the keyword classifier handles that.
crisis_posts = posts[posts['date'] >= '2026-02-20'].copy()
oil_posts = crisis_posts[crisis_posts['is_oil_related'] == True].copy().reset_index(drop=True)

print(f'Total posts in dataset: {len(posts)}')
print(f'Posts from Feb 20, 2026 onward: {len(crisis_posts)}')
print(f'Oil-related posts to classify with Gemini: {len(oil_posts)}')
print(f'Estimated Gemini time: ~{len(oil_posts) * 4 / 60:.0f} minutes')

In [ ]:
def classify_post_gemini(post_text, previous_text=None, post_timestamp=None):
    """
    Use Gemini to classify a Trump Truth Social post for market manipulation indicators.
    Uses a structured 5-dimension prompt with per-dimension reasoning.

    Returns dict with scores and reasoning.
    """
    prev_context = f'\nPREVIOUS POST (within 48h): "{previous_text}"' if previous_text else ''
    ts_context = f'\nTIMESTAMP: {post_timestamp}' if post_timestamp else ''

    prompt = f"""You are a forensic financial analyst. Classify the following Truth Social post from Donald Trump for oil market manipulation indicators.

POST: "{post_text}"{ts_context}{prev_context}

Score each dimension 0-100 and provide a 1-2 sentence reasoning for each:

1. ESCALATION_INTENT: Does this post threaten military action, sanctions, bombing, destruction, or economic harm that would push oil prices UP?
   0 = No escalation signals
   100 = Maximum escalation (explicit threats of war, bombing, blockade)

2. DE_ESCALATION_INTENT: Does this post suggest peace talks, deals, ceasefires, or reduced tensions that would push oil prices DOWN?
   0 = No de-escalation signals
   100 = Maximum de-escalation (explicit peace deal, withdrawal of threats)

3. FABRICATION_RISK: How likely is this specific claim to be false, unverifiable, or deliberately misleading?
   0 = Certainly true and verifiable
   100 = Certainly false (e.g., claiming talks that official sources deny)

4. MARKET_MOVER_PROBABILITY: Would a reasonable oil trader expect this post to move crude prices by >1%?
   Consider: vagueness, official-sounding language, reference to supply disruptions
   0 = No market impact expected
   100 = Definite major market-moving event

5. TIMING_SUSPICION: Is this post timed for maximum market impact?
   Consider: posted on weekend (markets closed, fear builds), posted Monday pre-market (thin liquidity),
   posted after-hours, posted during low-liquidity windows
   0 = Normal business hours, no timing pattern
   100 = Weekend + pre-market reversal (classic pump-and-dump timing)

6. CATEGORY: Classify as one of: ESCALATION, DE_ESCALATION, NEUTRAL, FABRICATION

Return ONLY valid JSON with NO markdown fences:
{{
    "escalation_intent": <0-100>,
    "escalation_reasoning": "<1-2 sentences>",
    "de_escalation_intent": <0-100>,
    "de_escalation_reasoning": "<1-2 sentences>",
    "fabrication_risk": <0-100>,
    "fabrication_reasoning": "<1-2 sentences>",
    "market_mover_probability": <0-100>,
    "market_mover_reasoning": "<1-2 sentences>",
    "timing_suspicion": <0-100>,
    "timing_reasoning": "<1-2 sentences>",
    "category": "<ESCALATION|DE_ESCALATION|NEUTRAL|FABRICATION>",
    "overall_reasoning": "<one sentence summary of manipulation assessment>"
}}"""

    try:
        response = client.models.generate_content(
            model='gemini-2.0-flash-lite',
            contents=prompt
        )

        text = response.text.strip()
        # Strip markdown fences if model ignores instruction
        if text.startswith('```'):
            text = text.split('\n', 1)[1]
            text = text.rsplit('```', 1)[0].strip()

        return json.loads(text)

    except json.JSONDecodeError as e:
        print(f'  JSON parse error: {e}')
        print(f'  Raw response: {response.text[:200]}')
        return None
    except Exception as e:
        print(f'  API error: {e}')
        return None

In [ ]:
# ============================================================
# PROMPT QUALITY TEST — Run this before full classification
# to verify the new prompt produces sensible scores.
#
# Expected results based on known ground truth:
#   March 23 post → fabrication_risk HIGH (>70)  [Iran denied it]
#                 → de_escalation_intent HIGH (>70)  [claimed peace]
#                 → market_mover_probability HIGH (>70)  [oil dropped 13%]
#   Escalation post → escalation_intent HIGH (>70), de_escalation LOW (<30)
#   Neutral post → escalation LOW (<20), market_mover LOW (<30)
# ============================================================

TEST_POSTS = [
    {
        "label": "March 23 fabrication (Iran denied this)",
        "text": "I AM PLEASED TO REPORT THAT THE UNITED STATES OF AMERICA, AND THE COUNTRY OF IRAN, HAVE REACHED A INITIAL AGREEMENT ON THE ENDING OF THE WAR IN THE MIDDLE EAST.",
        "timestamp": "2026-03-23 06:49:00",
        "expected": {"fabrication_risk": "HIGH (>70)", "de_escalation_intent": "HIGH (>70)", "market_mover_probability": "HIGH (>70)"}
    },
    {
        "label": "Escalation threat",
        "text": "If Iran doesn't FULLY OPEN, WITHOUT THREAT, the Strait of Hormuz, within 48 HOURS, the United States will have no choice but to take MAXIMUM PRESSURE action. This is your FINAL WARNING!!!",
        "timestamp": "2026-03-21 08:15:00",
        "expected": {"escalation_intent": "HIGH (>70)", "de_escalation_intent": "LOW (<30)", "fabrication_risk": "LOW (<30)"}
    },
    {
        "label": "Neutral / unrelated",
        "text": "The stock market is doing great. Unemployment is at record lows. America is winning again!",
        "timestamp": "2025-06-10 14:00:00",
        "expected": {"escalation_intent": "LOW (<20)", "market_mover_probability": "LOW (<30)"}
    },
]

if GEMINI_API_KEY != 'YOUR_GEMINI_KEY_HERE':
    print("=" * 60)
    print("PROMPT QUALITY TEST")
    print("=" * 60)
    for test in TEST_POSTS:
        print(f"\n📋 {test['label']}")
        result = classify_post_gemini(test['text'], post_timestamp=test['timestamp'])
        if result:
            print(f"   Category:               {result.get('category')}")
            print(f"   escalation_intent:      {result.get('escalation_intent')}")
            print(f"   de_escalation_intent:   {result.get('de_escalation_intent')}")
            print(f"   fabrication_risk:       {result.get('fabrication_risk')}")
            print(f"   market_mover_prob:      {result.get('market_mover_probability')}")
            print(f"   timing_suspicion:       {result.get('timing_suspicion')}")
            print(f"   reasoning:              {result.get('overall_reasoning')}")
            print(f"   Expected: {test['expected']}")
        else:
            print("   ❌ Classification failed")
    print("\n" + "=" * 60)
    print("If fabrication_risk is LOW for the March 23 post → prompt needs tuning.")
    print("=" * 60)
else:
    print("Set your Gemini API key in cell 1 to run the quality test.")

In [ ]:
import asyncio
import concurrent.futures

MAX_WORKERS = 10  # 10 concurrent Gemini requests

def _build_result(row, idx, classification):
    """Build result dict from a classification (or fallback)."""
    if classification:
        return {
            'post_id': row.get('post_id', idx),
            'date': str(row['date']),
            'timestamp': str(row['timestamp']),
            'post_text': row['post_text'][:200],
            # New 5-dimension schema
            'escalation_intent': classification.get('escalation_intent', 0),
            'escalation_reasoning': classification.get('escalation_reasoning', ''),
            'de_escalation_intent': classification.get('de_escalation_intent', 0),
            'de_escalation_reasoning': classification.get('de_escalation_reasoning', ''),
            'fabrication_risk': classification.get('fabrication_risk', 0),
            'fabrication_reasoning': classification.get('fabrication_reasoning', ''),
            'market_mover_probability': classification.get('market_mover_probability', 0),
            'market_mover_reasoning': classification.get('market_mover_reasoning', ''),
            'timing_suspicion': classification.get('timing_suspicion', 0),
            'timing_reasoning': classification.get('timing_reasoning', ''),
            'category': classification.get('category', 'NEUTRAL'),
            'reasoning': classification.get('overall_reasoning', ''),
        }
    else:
        return {
            'post_id': row.get('post_id', idx),
            'date': str(row['date']),
            'timestamp': str(row['timestamp']),
            'post_text': row['post_text'][:200],
            'escalation_intent': 0,
            'escalation_reasoning': '',
            'de_escalation_intent': 0,
            'de_escalation_reasoning': '',
            'fabrication_risk': 0,
            'fabrication_reasoning': '',
            'market_mover_probability': 0,
            'market_mover_reasoning': '',
            'timing_suspicion': 0,
            'timing_reasoning': '',
            'category': row.get('direction_keyword', 'NEUTRAL').upper(),
            'reasoning': 'API call failed, using keyword fallback',
        }

def _classify_one(args):
    """Classify a single post (runs in thread pool)."""
    idx, row, prev_text = args
    classification = classify_post_gemini(
        row['post_text'], prev_text, post_timestamp=str(row['timestamp'])
    )
    return idx, _build_result(row, idx, classification)

def classify_all_posts_concurrent(posts_df, max_workers=MAX_WORKERS,
                                  checkpoint_path='../data/processed/gemini_partial.csv',
                                  checkpoint_every=50):
    """
    Classify posts using a thread pool for concurrent Gemini API calls.
    Resumes from checkpoint if available.
    """
    results = {}
    start_idx = 0

    # Resume from checkpoint
    if os.path.exists(checkpoint_path):
        existing = pd.read_csv(checkpoint_path)
        for i, rec in existing.iterrows():
            results[i] = rec.to_dict()
        start_idx = len(results)
        print(f'Resuming from checkpoint at index {start_idx}')

    # Build work items (with previous-post context for oscillation scoring)
    work = []
    for i in range(start_idx, len(posts_df)):
        row = posts_df.iloc[i]
        prev_text = None
        if i > 0:
            prev_row = posts_df.iloc[i - 1]
            time_diff = (row['timestamp'] - prev_row['timestamp']).total_seconds() / 3600
            if time_diff <= 48:
                prev_text = prev_row['post_text']
        work.append((i, row, prev_text))

    total = len(posts_df)
    print(f'Classifying {len(work)} posts with {max_workers} concurrent workers...')

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_classify_one, w): w[0] for w in work}
        done_count = start_idx
        for future in concurrent.futures.as_completed(futures):
            idx, result = future.result()
            results[idx] = result
            done_count += 1

            if done_count % checkpoint_every == 0:
                sorted_results = [results[k] for k in sorted(results.keys())]
                pd.DataFrame(sorted_results).to_csv(checkpoint_path, index=False)
                print(f'  Checkpoint: {done_count}/{total} classified')

    sorted_results = [results[k] for k in sorted(results.keys())]
    return pd.DataFrame(sorted_results)

# Run classification
print(f'Posts to classify: {len(oil_posts)}')
print(f'Estimated time: ~{len(oil_posts) / MAX_WORKERS * 1.5 / 60:.1f} minutes '
      f'({MAX_WORKERS} concurrent requests)\n')

gemini_results = classify_all_posts_concurrent(oil_posts)
gemini_results.to_csv('../data/processed/gemini_classifications.csv', index=False)
print(f'\nDone! Saved {len(gemini_results)} classifications.')

In [ ]:
# Analysis of classification results
gemini_df = pd.read_csv('../data/processed/gemini_classifications.csv')

print('=== GEMINI CLASSIFICATION SUMMARY ===')
print(f'\nCategory distribution:')
print(gemini_df['category'].value_counts())

print(f'\nAverage scores:')
for col in ['direction_intent', 'verifiability', 'fabrication_risk', 
            'oscillation_risk', 'market_impact_intent']:
    if col in gemini_df.columns:
        print(f'  {col}: {gemini_df[col].mean():.1f}')

# Show highest manipulation intent posts
if 'market_impact_intent' in gemini_df.columns:
    print(f'\nTop 10 highest market impact intent:')
    top = gemini_df.nlargest(10, 'market_impact_intent')
    for _, row in top.iterrows():
        print(f'  [{row["date"]}] Score: {row["market_impact_intent"]} | {row["category"]} | {str(row["post_text"])[:80]}...')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Category distribution
cat_counts = gemini_df['category'].value_counts()
colors = [{'ESCALATION': '#d62728', 'DE_ESCALATION': '#2ca02c', 
           'NEUTRAL': '#7f7f7f', 'FABRICATION': '#ff7f0e'}.get(c, '#999')
          for c in cat_counts.index]
axes[0, 0].bar(cat_counts.index, cat_counts.values, color=colors)
axes[0, 0].set_title('Post Classification Distribution')
axes[0, 0].set_ylabel('Count')
axes[0, 0].tick_params(axis='x', rotation=15)

# Market impact intent histogram
if 'market_impact_intent' in gemini_df.columns:
    axes[0, 1].hist(gemini_df['market_impact_intent'], bins=20, color='#1f77b4', edgecolor='white')
    axes[0, 1].set_title('Distribution of Market Impact Intent Scores')
    axes[0, 1].set_xlabel('Market Impact Intent (0-100)')
    axes[0, 1].set_ylabel('Count')

# Fabrication risk histogram
if 'fabrication_risk' in gemini_df.columns:
    axes[1, 0].hist(gemini_df['fabrication_risk'], bins=20, color='#ff7f0e', edgecolor='white')
    axes[1, 0].set_title('Distribution of Fabrication Risk Scores')
    axes[1, 0].set_xlabel('Fabrication Risk (0-100)')
    axes[1, 0].set_ylabel('Count')

# Direction intent by category
if 'direction_intent' in gemini_df.columns:
    for cat in gemini_df['category'].unique():
        subset = gemini_df[gemini_df['category'] == cat]
        color = {'ESCALATION': '#d62728', 'DE_ESCALATION': '#2ca02c', 
                 'NEUTRAL': '#7f7f7f', 'FABRICATION': '#ff7f0e'}.get(cat, '#999')
        axes[1, 1].hist(subset['direction_intent'], bins=15, alpha=0.5, 
                        label=cat, color=color)
    axes[1, 1].set_title('Direction Intent by Category')
    axes[1, 1].set_xlabel('Direction Intent (0=Escalation, 100=De-escalation)')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].legend(fontsize=8)

plt.suptitle('Gemini AI Classification of Trump Oil-Related Posts', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/fig_gemini_classification.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nNext step: Run notebook 05 for fabrication and causality analysis.')